# 16 - The $Y$-$M$ scaling relation from the catalogue

We work with the Planck $Y$-$M$ relation, fitted **directly from the FLAMINGO
L2p8_m9 catalogue** (no halo-model / theory computation):

$$
\bar{E}^{-\beta}\left[\frac{D_A^2(z)\,\bar{Y}_{500}}{10^{-4}\,\mathrm{Mpc}^2}\right]
= Y^*\left(\frac{h}{0.7}\right)^{-2+\alpha}
\left[\frac{(1-b)\,M_{500}}{6\times10^{14}\,M_\odot}\right]^{\alpha}.
$$

**Key identity.** `Y_5R500c_Mpc2` already stores the *spherical* Compton-$Y$ in
physical area units, i.e. $D_A^2\,Y$, so the left bracket is `Y_5R500c_Mpc2 / 1e-4`.
**No per-cluster $D_A$ is needed**, only $E(z)$ (hmfast on a $z$-grid, interpolated).

**What is fixed and what is fitted.**
- $\beta = 2/3 \approx 0.66$ is **fixed** to the self-similar / Planck value. This is
  well justified: a free fit of **local** clusters ($z<0.3$) recovers $\beta\approx0.6$,
  right at self-similar (section 2). A *single* $E^\beta$ fitted over the full $0<z<3$
  gives a misleadingly low $\beta\simeq0.30$ because the evolution is not a pure power
  law in $E$ and the long high-$z$ lever arm (where it flattens) dominates.
- $Y^*$ is **fixed** to the Planck 2015 XXIV calibration, $\log_{10}Y^*=-0.19$,
  and the mass bias **$(1-b)$ is inferred** (section 3). $(1-b)$ and $Y^*$ are
  perfectly degenerate in-sample, so one must be fixed to obtain the other.
- $\alpha$ (the mass slope) and the intrinsic scatter are **fitted** from the data.

**Cosmology.** $h=0.681$ (FLAMINGO D3A); the $(h/0.7)^{-2+\alpha}$ factor is a fixed
constant. LHS uses `Y_5R500c` (within $5R_{500}$), converted to Planck's $Y_{500}$
when inferring $(1-b)$ via the A10 spherical aperture ratio.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")  # E(z) on a grid only; no GPU needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from scipy.integrate import quad

from flamingo.catalogue import frame  # hmfast D3A cosmology (h=0.681)

CAT = "/scratch/scratch-lxu/flamingo_repo/data/hydro_L2p8m9/catalogue/halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv"
df = pd.read_csv(CAT, comment="#")
print(f"N clusters = {len(df):,}")

z = df["z"].to_numpy()
M500 = df["M_500c_Msun"].to_numpy()       # true SO mass, Msun
Y = df["Y_5R500c_Mpc2"].to_numpy()        # = D_A^2 * Y_500, Mpc^2

# E(z) from hmfast on a grid, then interpolate (avoids 1.5M GPU calls)
zgrid = np.linspace(0.0, z.max() + 1e-3, 600)
E = np.interp(z, zgrid, frame.efunc(zgrid))

h = frame._h                 # 0.681
LOG_h07 = np.log10(h / 0.7)
M_PIVOT = 6e14               # Msun
Y_UNIT = 1e-4                # Mpc^2
BETA = 2.0 / 3.0             # FIXED: self-similar / Planck evolution

logM = np.log10(M500 / M_PIVOT)
logE = np.log10(E)
y = np.log10(Y / Y_UNIT)
print(f"h = {h}, log10(h/0.7) = {LOG_h07:.5f}, fixed beta = {BETA:.4f}")


N clusters = 1,555,542


I0000 00:00:1782460251.820443 2556195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460251.820797 2556195 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1782460252.779856 2556195 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460252.780181 2556195 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


h = 0.6809999999999999, log10(h/0.7) = -0.01195, fixed beta = 0.6667


## 1. Fit $\alpha$ and the scatter (with $\beta=2/3$ fixed)

With the self-similar evolution removed, $w \equiv \log_{10}(Y_{5R500c}/10^{-4})
- \tfrac23\log_{10}E$ is a pure power law in mass:
$w = c_0 + \alpha\,\log_{10}(M/6\times10^{14})$, with
$c_0 = \log_{10}Y^*_{\rm FL} + (\alpha-2)\log_{10}(h/0.7)$. We fit per-cluster and
binned in mass. ($Y^*_{\rm FL}$ here is FLAMINGO's own amplitude for `Y_5R500c`
with $(1-b)=1$; it is *not* the Planck $Y^*$, which we use only in section 3.)

In [2]:
w = y - BETA * logE        # self-similar evolution removed

def fit_slope(logM, w, wt=None):
    A = np.column_stack([np.ones_like(logM), logM])
    if wt is not None:
        sw = np.sqrt(wt)
        coef, *_ = np.linalg.lstsq(A * sw[:, None], w * sw, rcond=None)
    else:
        coef, *_ = np.linalg.lstsq(A, w, rcond=None)
    c0, alpha = coef
    resid = w - A @ coef
    YstarFL = 10.0 ** (c0 - (alpha - 2.0) * LOG_h07)
    return dict(c0=c0, alpha=alpha, YstarFL=YstarFL,
                sigma=float(np.std(resid)), sigma_ln=float(np.std(resid) * np.log(10.0)),
                resid=resid)

fit_pc = fit_slope(logM, w)
print("Per-cluster (beta=2/3 fixed, (1-b)=1):")
print(f"  alpha           = {fit_pc['alpha']:.4f}   (self-similar 5/3 = {5/3:.3f})")
print(f"  Y*_FL [Y_5R500c]= {fit_pc['YstarFL']:.4f}   (log10 = {np.log10(fit_pc['YstarFL']):.4f})")
print(f"  pivot amplitude c0 = {fit_pc['c0']:.4f}")
print(f"  scatter         = {fit_pc['sigma']:.4f} dex  ({fit_pc['sigma_ln']:.4f} in ln Y)")

# binned in mass
logM_abs = np.log10(M500)
edges = np.linspace(logM_abs.min(), logM_abs.max(), 16)
idx = np.clip(np.digitize(logM_abs, edges) - 1, 0, len(edges) - 2)
bM, bw, bn = [], [], []
for a in range(len(edges) - 1):
    s = idx == a
    if s.sum() < 50:
        continue
    bM.append(logM[s].mean()); bw.append(w[s].mean()); bn.append(int(s.sum()))
bM = np.array(bM); bw = np.array(bw); bn = np.array(bn, float)
fit_bin = fit_slope(bM, bw, wt=bn)
print(f"\nBinned ({len(bn)} mass bins): alpha = {fit_bin['alpha']:.4f}, "
      f"Y*_FL = {fit_bin['YstarFL']:.4f}")

Per-cluster (beta=2/3 fixed, (1-b)=1):
  alpha           = 1.6117   (self-similar 5/3 = 1.667)
  Y*_FL [Y_5R500c]= 0.8626   (log10 = -0.0642)
  pivot amplitude c0 = -0.0596
  scatter         = 0.1222 dex  (0.2813 in ln Y)

Binned (14 mass bins): alpha = 1.6113, Y*_FL = 0.8620


## 2. Subsample investigation: massive, small, local, distant

We refit in subsamples to test where the relation holds. For each we report the
**free** $(\alpha,\beta)$, the slope $\alpha$ and inferred $(1-b)$ at fixed
$\beta=2/3$. The headline is the evolution exponent:

- **Local clusters ($z<0.3$) recover $\beta\approx0.6$**, near self-similar $2/3$ -
  small local clusters reach $0.62$. So the self-similar evolution is correct
  locally, validating the $\beta=2/3$ choice.
- The low global $\beta\simeq0.30$ comes entirely from the **distant** population
  ($z>1$ gives $\beta\approx0.2$): $Y$ stops tracking $E^{2/3}$ at high $z$, so a
  single power law averaged over $0<z<3$ understates the local evolution.
- $(1-b)$ is stable at $\sim0.83$-$0.89$ across mass and is mildly lower for distant
  clusters. The **Planck-comparable massive+local** subsample gives $(1-b)\approx0.87$.

In [3]:
def subfit(mask):
    n = int(mask.sum())
    cf = np.linalg.lstsq(np.column_stack([np.ones(n), logM[mask], logE[mask]]), y[mask], rcond=None)[0]
    c0, al = np.linalg.lstsq(np.column_stack([np.ones(n), logM[mask]]), w[mask], rcond=None)[0]
    return n, cf[1], cf[2], al, c0   # N, alpha_free, beta_free, alpha(b=2/3), pivot c0

subs = [
    ("ALL",                       np.ones_like(z, bool)),
    ("small   M<1e14",            logM_abs < 14.0),
    ("medium  1-3e14",            (logM_abs >= 14.0) & (logM_abs < 14.5)),
    ("massive M>3e14",            logM_abs >= 14.5),
    ("local   z<0.3",             z < 0.3),
    ("local small  z<0.3 M<1e14", (z < 0.3) & (logM_abs < 14.0)),
    ("local massive z<0.3 M>3e14",(z < 0.3) & (logM_abs >= 14.5)),
    ("distant z>1   M>1e14",      (z > 1.0) & (logM_abs >= 14.0)),
]
print(f"{'subsample':28s}{'N':>9s}{'a_free':>8s}{'b_free':>8s}{'a(b=2/3)':>9s}{'(1-b)':>8s}")
sub_results = {}
for name, m in subs:
    n, af, bf, a23, c0 = subfit(m)
    omb = 10.0 ** ((c0 - (np.log10(1.8141) - 0.19 + (1.79 - 2) * LOG_h07)) / 1.79)
    sub_results[name] = (n, af, bf, a23, omb)
    print(f"{name:28s}{n:9d}{af:8.3f}{bf:8.3f}{a23:9.3f}{omb:8.3f}")

subsample                           N  a_free  b_free a(b=2/3)   (1-b)


ALL                           1555542   1.570   0.296    1.612   0.845


small   M<1e14                1192738   1.587   0.301    1.629   0.864
medium  1-3e14                 345758   1.553   0.276    1.596   0.835


massive M>3e14                  17046   1.540   0.256    1.577   0.829
local   z<0.3                   91860   1.602   0.570    1.602   0.887
local small  z<0.3 M<1e14       60699   1.630   0.622    1.630   0.917
local massive z<0.3 M>3e14       2857   1.561   0.262    1.562   0.867
distant z>1   M>1e14            87974   1.539   0.196    1.564   0.761


## 3. Fix $Y^*$ (Planck 2015 XXIV), infer $(1-b)$

Planck measures $Y_{500}$ (within $R_{500}$); the catalogue stores $Y_{5R500c}$
(within $5R_{500}$). We convert with the **spherical aperture ratio**
$\rho=Y(<5R_{500})/Y(<R_{500})$ of the A10 UPP. With $\beta=2/3$ and Planck's
$Y^*_{\rm P}=10^{-0.19}$, $\alpha_{\rm P}=1.79$ fixed, the mass bias is the shift that
places FLAMINGO's pivot amplitude onto the Planck relation:
$$
(1-b)=\left[\frac{10^{c_0}}{\rho\,Y^*_{\rm P}\,(h/0.7)^{\alpha_{\rm P}-2}}\right]^{1/\alpha_{\rm P}},
$$
with $c_0$ the FLAMINGO pivot amplitude from section 1 (for $Y_{5R500c}$, $\beta=2/3$).

In [4]:
# A10 UPP spherical aperture ratio
P0, c500, a_g, b_g, g_g = 8.130, 1.156, 1.0620, 5.4807, 0.3292
p = lambda x: P0 / ((c500 * x) ** g_g * (1.0 + (c500 * x) ** a_g) ** ((b_g - g_g) / a_g))
rho = quad(lambda x: x * x * p(x), 0, 5)[0] / quad(lambda x: x * x * p(x), 0, 1)[0]

logYstar_P, alpha_P = -0.19, 1.79          # Planck 2015 XXIV (fixed)
logamp_P = np.log10(rho) + logYstar_P + (alpha_P - 2.0) * LOG_h07   # Planck pivot, (1-b)=1

def infer_1mb(c0):
    return 10.0 ** ((c0 - logamp_P) / alpha_P)

omb_pc = infer_1mb(fit_pc["c0"])
omb_bin = infer_1mb(fit_bin["c0"])
print(f"aperture ratio rho = {rho:.4f}")
print(f"Planck anchor: log10 Y*_P = {logYstar_P}, alpha_P = {alpha_P}, beta = 2/3 (fixed)")
omb_localmassive = sub_results["local massive z<0.3 M>3e14"][4]
print(f"  (1-b)  [per-cluster, all]      = {omb_pc:.4f}   ->  b = {1 - omb_pc:.4f}")
print(f"  (1-b)  [binned, all]           = {omb_bin:.4f}   ->  b = {1 - omb_bin:.4f}")
print(f"  (1-b)  [massive+local, Planck-like] = {omb_localmassive:.4f}   ->  b = {1 - omb_localmassive:.4f}")

aperture ratio rho = 1.8141
Planck anchor: log10 Y*_P = -0.19, alpha_P = 1.79, beta = 2/3 (fixed)
  (1-b)  [per-cluster, all]      = 0.8452   ->  b = 0.1548
  (1-b)  [binned, all]           = 0.8449   ->  b = 0.1551
  (1-b)  [massive+local, Planck-like] = 0.8673   ->  b = 0.1327


## 4. Fix only $Y^*$: jointly infer $\alpha$, $\beta$, $(1-b)$

Pinning $Y^*$ to Planck breaks the $Y^*$-$(1-b)$ degeneracy, so a **single** OLS on
$[1,\log_{10}M,\log_{10}E]$ identifies all three at once: $\alpha$ (the $\log_{10}M$
slope), $\beta$ (the $\log_{10}E$ slope, now **free**), and $(1-b)$ from the intercept
$c_0$ using the fitted $\alpha$,
$$
(1-b)=10^{\,[\,c_0-\log_{10}\rho-\log_{10}Y^*_{\rm P}-(\alpha-2)\log_{10}(h/0.7)\,]/\alpha}.
$$
Globally this gives $\beta\simeq0.30$ (the single-power-law value); recall from section 2
that local clusters give $\beta\approx0.6$. $(1-b)$ is stable at $\sim0.88$.

In [5]:
def joint_fix_Ystar(mask):
    n = int(mask.sum())
    A = np.column_stack([np.ones(n), logM[mask], logE[mask]])
    c0, al, be = np.linalg.lstsq(A, y[mask], rcond=None)[0]
    resid = y[mask] - A @ np.array([c0, al, be])
    log1mb = (c0 - np.log10(rho) - logYstar_P - (al - 2.0) * LOG_h07) / al
    return dict(N=n, alpha=al, beta=be, omb=10.0 ** log1mb, sigma=float(np.std(resid)))

jf = joint_fix_Ystar(np.ones_like(z, bool))
print("Fix Y* = Planck (10^-0.19), infer alpha, beta, (1-b) jointly:")
print(f"  alpha   = {jf['alpha']:.4f}")
print(f"  beta    = {jf['beta']:.4f}   (global single power law; local z<0.3 -> ~0.6)")
print(f"  (1-b)   = {jf['omb']:.4f}   ->  b = {1 - jf['omb']:.4f}")
print(f"  scatter = {jf['sigma']:.4f} dex")

print(f"\n{'subsample':28s}{'N':>9s}{'alpha':>8s}{'beta':>8s}{'(1-b)':>8s}")
for name, m in subs:
    r = joint_fix_Ystar(m)
    print(f"{name:28s}{r['N']:9d}{r['alpha']:8.3f}{r['beta']:8.3f}{r['omb']:8.3f}")

Fix Y* = Planck (10^-0.19), infer alpha, beta, (1-b) jointly:
  alpha   = 1.5699
  beta    = 0.2962   (global single power law; local z<0.3 -> ~0.6)
  (1-b)   = 0.8770   ->  b = 0.1230
  scatter = 0.1156 dex

subsample                           N   alpha    beta   (1-b)
ALL                           1555542   1.570   0.296   0.877


small   M<1e14                1192738   1.587   0.301   0.899


medium  1-3e14                 345758   1.553   0.276   0.868
massive M>3e14                  17046   1.540   0.256   0.865
local   z<0.3                   91860   1.602   0.570   0.877
local small  z<0.3 M<1e14       60699   1.630   0.622   0.909
local massive z<0.3 M>3e14       2857   1.561   0.262   0.870
distant z>1   M>1e14            87974   1.539   0.196   0.885


## 5. Figure

In [6]:
c0, alpha = fit_pc["c0"], fit_pc["alpha"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

hb = ax1.hexbin(logM_abs, w, gridsize=70, bins="log", cmap="viridis", mincnt=1)
mm = np.linspace(logM_abs.min(), logM_abs.max(), 100)
ax1.plot(mm, c0 + alpha * (mm - np.log10(M_PIVOT)), "r-", lw=2,
         label=fr"fit: $\alpha={alpha:.3f}$, $\beta=2/3$ (fixed)")
# binned mean +/- standard error of the mean
be = np.linspace(logM_abs.min(), logM_abs.max(), 16)
bc, bmw, bsew = [], [], []
for i in range(len(be) - 1):
    s = (logM_abs >= be[i]) & (logM_abs < be[i + 1])
    if s.sum() < 50:
        continue
    bc.append(0.5 * (be[i] + be[i + 1])); bmw.append(w[s].mean()); bsew.append(w[s].std() / np.sqrt(s.sum()))
ax1.errorbar(bc, bmw, yerr=bsew, fmt="ks", ms=4, capsize=2, label="binned mean (SE)")
ax1.axvline(np.log10(M_PIVOT), color="k", ls=":", lw=1, label=r"pivot $6\times10^{14}$")
ax1.set_xlabel(r"$\log_{10}\,M_{500c}\ [M_\odot]$")
ax1.set_ylabel(r"$\log_{10}\left[E^{-2/3}\,Y_{5R500c}/10^{-4}\,\mathrm{Mpc}^2\right]$")
ax1.set_title(r"$Y$-$M$ relation (self-similar $E^{-2/3}$ removed)")
ax1.legend(loc="upper left", fontsize=9)
fig.colorbar(hb, ax=ax1, label=r"$\log_{10}N$")

ax2.hexbin(z, fit_pc["resid"], gridsize=70, bins="log", cmap="cividis", mincnt=1)
ax2.axhline(0.0, color="r", lw=1.5)
zb = np.linspace(z.min(), z.max(), 25); zc = 0.5 * (zb[1:] + zb[:-1])
med = np.full(len(zc), np.nan); mede = np.full(len(zc), np.nan)
for i in range(len(zc)):
    s = (z >= zb[i]) & (z < zb[i + 1])
    if s.sum() > 10:
        med[i] = np.median(fit_pc["resid"][s])
        mede[i] = 1.2533 * np.std(fit_pc["resid"][s]) / np.sqrt(s.sum())   # SE of the median
ax2.errorbar(zc, med, yerr=mede, fmt="w.-", lw=1.5, ms=5, capsize=2, label="median residual (SE)")
ax2.set_xlabel(r"$z$"); ax2.set_ylabel(r"residual at fixed $\beta=2/3$")
ax2.set_title(r"residual vs $z$ (flat at low $z$; high-$z$ flattening)")
ax2.legend(loc="upper right", fontsize=9)

fig.tight_layout()
figdir = "/scratch/scratch-lxu/flamingo_repo/autoresearch/figures"
fig.savefig(f"{figdir}/nb16_yM_scaling.pdf"); fig.savefig(f"{figdir}/nb16_yM_scaling.png"), dpi=300)
plt.show()
print("saved nb16_yM_scaling.{pdf,png}")

saved nb16_yM_scaling.{pdf,png}


## 6. Evolution of $\alpha$ and $\beta$ with mass and redshift

We measure each slope where it has lever arm:
- **$\alpha$ vs $z$**: fit the mass slope in each $z$-bin ($\beta=2/3$ fixed).
  **$\alpha$ vs $M$**: running local slope $d\langle w\rangle/d\langle\log_{10}M\rangle$.
- **$\beta$ vs $M$**: fit $[\,1,\log_{10}M,\log_{10}E\,]$ in each mass bin (full $z$ lever).
  **$\beta$ vs $z$**: running slope $\beta_{\rm eff}(z)=d\langle Q\rangle/d\langle\log_{10}E\rangle$
  of the mass-corrected $Q=\log_{10}(Y/10^{-4})-\alpha\log_{10}M$.

**The headline is $\beta(z)$: it falls from $\approx0.55$ near $z=0.25$ to $\approx0.2$ by
$z\sim1.8$** - the evolution is not a single power law in $E$. The mass bias $(1-b)$ is a single
relation-level parameter (the amplitude shift, derived from the fit intercept using the
relation's own fitted $\alpha$); its dependence on the $\beta$ fixing is shown in section 6b,
not here.

In [7]:
# --- Evolution of alpha and beta with M and z; (1-b) uses the FITTED slope ---
LN10 = np.log(10.0)
K_P = np.log10(rho) + logYstar_P     # Planck Y_500 amplitude at pivot, (1-b)=1

def ols(X, yv):
    b, *_ = np.linalg.lstsq(X, yv, rcond=None)
    resid = yv - X @ b
    n, p = X.shape
    cov = (resid @ resid / (n - p)) * np.linalg.inv(X.T @ X)
    return b, np.sqrt(np.diag(cov))

def eqbins(v, nb):
    return np.quantile(v, np.linspace(0, 1, nb + 1))

def omb_from_fit(c0, al, c0se=0.0, alse=0.0):
    """(1-b) at the pivot from amplitude c0 and the relation's OWN fitted slope al."""
    log1mb = (c0 - K_P - (al - 2.0) * LOG_h07) / al
    o = 10.0 ** log1mb
    se = LN10 * o * np.sqrt((c0se / al) ** 2 + (log1mb / al * alse) ** 2)
    return o, se

logM_abs = np.log10(M500)
# alpha vs z: per z-bin mass slope (beta=2/3), fit w ~ [1, logM]
ze = eqbins(z, 12)
z_med, alpha_z, alpha_z_se = [], [], []
for i in range(len(ze) - 1):
    s = (z >= ze[i]) & (z < ze[i + 1])
    b, se = ols(np.column_stack([np.ones(s.sum()), logM[s]]), w[s])
    z_med.append(np.median(z[s])); alpha_z.append(b[1]); alpha_z_se.append(se[1])
z_med, alpha_z, alpha_z_se = map(np.array, (z_med, alpha_z, alpha_z_se))

# beta vs M: per mass-bin evolution slope, fit y ~ [1, logM, logE]
me = eqbins(logM_abs, 8)
M_med, beta_M, beta_M_se = [], [], []
for i in range(len(me) - 1):
    s = (logM_abs >= me[i]) & (logM_abs < me[i + 1])
    b, se = ols(np.column_stack([np.ones(s.sum()), logM[s], logE[s]]), y[s])
    M_med.append(np.median(logM_abs[s])); beta_M.append(b[2]); beta_M_se.append(se[2])
M_med, beta_M, beta_M_se = map(np.array, (M_med, beta_M, beta_M_se))

def running_slope(xc, mean, se):
    g = np.gradient(mean, xc); ge = np.zeros_like(g)
    ge[1:-1] = np.sqrt(se[2:] ** 2 + se[:-2] ** 2) / (xc[2:] - xc[:-2])
    ge[0] = np.sqrt(se[1] ** 2 + se[0] ** 2) / (xc[1] - xc[0])
    ge[-1] = np.sqrt(se[-1] ** 2 + se[-2] ** 2) / (xc[-1] - xc[-2])
    return g, ge

# alpha vs M: running d<w>/d<logM>
mf = eqbins(logM_abs, 12); Mc2, wm, wse, lMm = [], [], [], []
for i in range(len(mf) - 1):
    s = (logM_abs >= mf[i]) & (logM_abs < mf[i + 1])
    Mc2.append(np.median(logM_abs[s])); wm.append(w[s].mean()); wse.append(w[s].std() / np.sqrt(s.sum())); lMm.append(logM[s].mean())
Mc2 = np.array(Mc2); alpha_M, alpha_M_se = running_slope(np.array(lMm), np.array(wm), np.array(wse))

# beta vs z: running d<Q>/d<logE>, Q = y - alpha*logM (mass-corrected)
alpha_ref = fit_pc["alpha"]; Q = y - alpha_ref * logM
zf = eqbins(z, 16); zc2, Qm, Qse, lEm = [], [], [], []
for i in range(len(zf) - 1):
    s = (z >= zf[i]) & (z < zf[i + 1])
    zc2.append(np.median(z[s])); Qm.append(Q[s].mean()); Qse.append(Q[s].std() / np.sqrt(s.sum())); lEm.append(logE[s].mean())
zc2 = np.array(zc2); beta_z, beta_z_se = running_slope(np.array(lEm), np.array(Qm), np.array(Qse))
print(f"alpha(z) {alpha_z.min():.3f}-{alpha_z.max():.3f}; beta(z) {beta_z[0]:.3f} -> {beta_z[-1]:.3f}")

alpha(z) 1.556-1.599; beta(z) 0.556 -> 0.202


In [8]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7), sharex="col")
C, HL = "#0072B2", "#D55E00"
ax[0, 0].fill_between(Mc2, alpha_M - alpha_M_se, alpha_M + alpha_M_se, color=C, alpha=0.25)
ax[0, 0].plot(Mc2, alpha_M, "o-", color=C, ms=4)
ax[0, 0].axhline(5/3, ls="--", color="k", lw=1, label="self-similar 5/3")
ax[0, 0].set_ylabel(r"$\alpha$"); ax[0, 0].set_title("vs mass"); ax[0, 0].legend(fontsize=8)
ax[0, 1].errorbar(z_med, alpha_z, yerr=alpha_z_se, fmt="s-", color=C, ms=4, capsize=2)
ax[0, 1].axhline(5/3, ls="--", color="k", lw=1); ax[0, 1].set_title("vs redshift")
ax[1, 0].errorbar(M_med, beta_M, yerr=beta_M_se, fmt="o-", color=C, ms=4, capsize=2)
ax[1, 0].axhline(2/3, ls="--", color="k", lw=1, label="self-similar 2/3")
ax[1, 0].set_ylabel(r"$\beta$"); ax[1, 0].set_xlabel(r"$\log_{10}M_{500c}\ [M_\odot]$"); ax[1, 0].legend(fontsize=8)
ax[1, 1].fill_between(zc2, beta_z - beta_z_se, beta_z + beta_z_se, color=HL, alpha=0.3)
ax[1, 1].plot(zc2, beta_z, "s-", color=HL, lw=2, ms=6)
ax[1, 1].axhline(2/3, ls="--", color="k", lw=1, label="self-similar 2/3")
ax[1, 1].set_ylabel(r"$\beta_{\rm eff}(z)$"); ax[1, 1].set_xlabel(r"$z$"); ax[1, 1].legend(fontsize=8)
ax[1, 1].annotate("evolution flattens\nat high z", xy=(zc2[-3], beta_z[-3]), xytext=(0.9, 0.45),
                  fontsize=9, arrowprops=dict(arrowstyle="->"))
for a in ax.ravel():
    a.grid(alpha=0.25)
fig.suptitle(r"Evolution of $\alpha$ and $\beta$ with mass and redshift (bands/bars = OLS uncertainty)", fontsize=12)
fig.tight_layout()
figdir = "/scratch/scratch-lxu/flamingo_repo/autoresearch/figures"
fig.savefig(f"{figdir}/nb16_param_evolution.pdf", dpi=300)
fig.savefig(f"{figdir}/nb16_param_evolution.png"), dpi=300)
plt.show(); print("saved nb16_param_evolution.{pdf,png}")

saved nb16_param_evolution.{pdf,png}


## 6b. Method-comparison plots

Sections 2-4 reported numbers only; here we plot them so every framing has a figure. The
mass bias $(1-b)$ is always taken **at the pivot mass** from the fit amplitude using the
relation's **own fitted $\alpha$** (no per-cluster values, no Planck-slope projection), so all
figures are mutually consistent.

- **`nb16_subsample_fits`**: $\alpha$, $\beta$, and $(1-b)$ across subsamples, comparing the
  free fit / $\beta=2/3$ fixed / joint framings (error bars = OLS SE).
- **`nb16_evolution_fixings`**: the **same** $(1-b)$ evolves differently depending on what is
  fixed. It **drifts down** with $z$ when $\beta=2/3$ is imposed but is **flat** when $\beta$ is
  free; the free $\beta_{\rm eff}(z)$ falls from $\approx0.6$ locally to $\approx0.2$ at high $z$.

In [9]:
# --- Sec 2 & 4 visualised: subsample fits under different fixings (fitted-alpha (1-b)) ---
figdir = "/scratch/scratch-lxu/flamingo_repo/autoresearch/figures"
names, al_, ale_, a23_, a23e_, be_, bee_, omb23_, omb23e_, ombf_, ombfe_ = ([] for _ in range(11))
for name, m in subs:
    n = int(m.sum())
    cf, sf = ols(np.column_stack([np.ones(n), logM[m], logE[m]]), y[m])  # free: c0, alpha, beta
    cw, sw = ols(np.column_stack([np.ones(n), logM[m]]), w[m])            # beta=2/3 fixed: c0, alpha
    names.append(name)
    al_.append(cf[1]); ale_.append(sf[1]); be_.append(cf[2]); bee_.append(sf[2])
    a23_.append(cw[1]); a23e_.append(sw[1])
    o23, o23e = omb_from_fit(cw[0], cw[1], sw[0], sw[1])   # (1-b) | beta=2/3, fitted alpha
    of, ofe = omb_from_fit(cf[0], cf[1], sf[0], sf[1])     # (1-b) | beta free, fitted alpha
    omb23_.append(o23); omb23e_.append(o23e); ombf_.append(of); ombfe_.append(ofe)
yy = np.arange(len(names))[::-1]
C, HL = "#0072B2", "#D55E00"
fig, ax = plt.subplots(1, 3, figsize=(13, 4.6), sharey=True)
ax[0].errorbar(al_, yy, xerr=ale_, fmt="o", color=C, capsize=2, label=r"free fit")
ax[0].errorbar(a23_, yy, xerr=a23e_, fmt="s", color=HL, mfc="none", capsize=2, label=r"$\beta=2/3$ fixed")
ax[0].axvline(5/3, ls="--", color="k", lw=1, label="self-similar 5/3")
ax[0].set_xlabel(r"$\alpha$"); ax[0].set_title("mass slope")
ax[0].set_yticks(yy); ax[0].set_yticklabels(names, fontsize=8); ax[0].legend(fontsize=7)
ax[1].errorbar(be_, yy, xerr=bee_, fmt="o", color=C, capsize=2, label=r"free $\beta$")
ax[1].axvline(2/3, ls="--", color="k", lw=1, label="self-similar 2/3")
ax[1].set_xlabel(r"$\beta$"); ax[1].set_title("evolution exponent (free)"); ax[1].legend(fontsize=7)
ax[2].errorbar(omb23_, yy, xerr=omb23e_, fmt="o", color=C, capsize=2, label=r"$\beta=2/3$ fixed")
ax[2].errorbar(ombf_, yy, xerr=ombfe_, fmt="s", color=HL, mfc="none", capsize=2, label=r"$\beta$ free (joint)")
ax[2].axvline(1.0, ls=":", color="k", lw=1)
ax[2].set_xlabel(r"$(1-b)$"); ax[2].set_title("hydrostatic bias"); ax[2].legend(fontsize=7)
for a in ax:
    a.grid(alpha=0.25)
fig.suptitle("Subsample fits: mass slope, evolution, and bias under different fixings "
             "(fitted-alpha (1-b); error bars = OLS SE)", fontsize=11)
fig.tight_layout()
fig.savefig(f"{figdir}/nb16_subsample_fits.pdf", dpi=300)
fig.savefig(f"{figdir}/nb16_subsample_fits.png"), dpi=300)
plt.show(); print("saved nb16_subsample_fits.{pdf,png}")

saved nb16_subsample_fits.{pdf,png}


In [10]:
# --- (1-b) vs z under two beta fixings (fitted-alpha; the adopted ground-truth method) ---
figdir = "/scratch/scratch-lxu/flamingo_repo/autoresearch/figures"
beta_glob = jf["beta"]            # global free evolution exponent (Sec 4)
w_free = y - beta_glob * logE

def omb_z(wv):
    """per z-bin (1-b) from fit amplitude + the relation's OWN fitted slope.
    central = bin value; error bars = 16-84% per-cluster scatter (the SE of the mean is
    ~0.003, smaller than a marker)."""
    ze = eqbins(z, 12); zc, cen, lo, hi = [], [], [], []
    for i in range(len(ze) - 1):
        s = (z >= ze[i]) & (z < ze[i + 1]); n = int(s.sum())
        cf, _ = ols(np.column_stack([np.ones(n), logM[s]]), wv[s])
        res = wv[s] - (cf[0] + cf[1] * logM[s])
        cen.append(omb_from_fit(cf[0], cf[1])[0])
        lo.append(omb_from_fit(cf[0] + np.percentile(res, 16), cf[1])[0])
        hi.append(omb_from_fit(cf[0] + np.percentile(res, 84), cf[1])[0])
        zc.append(np.median(z[s]))
    return (np.array(v) for v in (zc, cen, lo, hi))

zc_, f_c, f_lo, f_hi = omb_z(w)         # beta = 2/3 fixed
_, g_c, g_lo, g_hi = omb_z(w_free)      # beta free-global
print(f"(1-b): fixed {f_c.mean():.3f} (all-z mean), free {g_c.mean():.3f}")
C, HL = "#0072B2", "#D55E00"
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 4.4))
axA.errorbar(zc_, f_c, yerr=[f_c - f_lo, f_hi - f_c], fmt="s-", color=C, capsize=3, elinewidth=1,
             label=r"$\beta=2/3$ fixed (drifts down)")
axA.errorbar(zc_ + 0.015, g_c, yerr=[g_c - g_lo, g_hi - g_c], fmt="o-", color=HL, capsize=3, elinewidth=1,
             label=fr"$\beta={beta_glob:.2f}$ free-global (little drift)")
axA.axhline(1.0, ls=":", color="k", lw=1)
axA.set_xlabel(r"$z$"); axA.set_ylabel(r"$(1-b)$  hydrostatic mass bias")
axA.set_title(r"$(1-b)$ evolution depends on the $\beta$ fixing")
axA.legend(fontsize=8, title="error bars: 16-84% cluster scatter", title_fontsize=7)
axB.fill_between(zc2, beta_z - beta_z_se, beta_z + beta_z_se, color=HL, alpha=0.3)
axB.errorbar(zc2, beta_z, yerr=beta_z_se, fmt="o-", color=HL, capsize=2,
             label=r"$\beta_{\rm eff}(z)$ running slope")
axB.axhline(2/3, ls="--", color="k", lw=1, label="self-similar 2/3")
axB.set_xlabel(r"$z$"); axB.set_ylabel(r"$\beta_{\rm eff}(z)$")
axB.set_title(r"evolution exponent declines with $z$"); axB.legend(fontsize=8)
for a in (axA, axB):
    a.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(f"{figdir}/nb16_evolution_fixings.pdf", dpi=300)
fig.savefig(f"{figdir}/nb16_evolution_fixings.png"), dpi=300)
plt.show(); print("saved nb16_evolution_fixings.{pdf,png}")

(1-b): fixed 0.779 (all-z mean), free 0.873


saved nb16_evolution_fixings.{pdf,png}


## 7. Summary

In [11]:
print("="*60)
print("Y-M scaling from the FLAMINGO catalogue (Y_5R500c = D_A^2 Y_500)")
print("beta = 2/3 FIXED (self-similar);  Y* FIXED to Planck 2015 XXIV")
print("="*60)
print(f"  alpha (fitted)          = {fit_pc['alpha']:.4f}   (binned {fit_bin['alpha']:.4f})")
print(f"  scatter                 = {fit_pc['sigma']:.4f} dex")
print(f"  Y*_FL [Y_5R500c,(1-b)=1]= {fit_pc['YstarFL']:.4f}")
print(f"  beta: local z<0.3 free  = {sub_results['local   z<0.3'][2]:.3f} (~self-similar 2/3)")
print(f"        global free       = {sub_results['ALL'][2]:.3f} (high-z power-law breakdown)")
print(f"  --- inferred bias vs Planck Y*=10^-0.19 (beta=2/3 fixed) ---")
print(f"  (1-b)  all clusters     = {omb_pc:.4f}   ->  b = {1 - omb_pc:.4f}")
print(f"  (1-b)  massive + local  = {omb_localmassive:.4f}   ->  b = {1 - omb_localmassive:.4f}")
print(f"  --- fix Y* only, jointly infer alpha,beta,(1-b) (beta free) ---")
print(f"  alpha = {jf['alpha']:.4f},  beta = {jf['beta']:.4f},  (1-b) = {jf['omb']:.4f}  (b = {1 - jf['omb']:.4f})")
print("="*60)

Y-M scaling from the FLAMINGO catalogue (Y_5R500c = D_A^2 Y_500)
beta = 2/3 FIXED (self-similar);  Y* FIXED to Planck 2015 XXIV
  alpha (fitted)          = 1.6117   (binned 1.6113)
  scatter                 = 0.1222 dex
  Y*_FL [Y_5R500c,(1-b)=1]= 0.8626
  beta: local z<0.3 free  = 0.570 (~self-similar 2/3)
        global free       = 0.296 (high-z power-law breakdown)
  --- inferred bias vs Planck Y*=10^-0.19 (beta=2/3 fixed) ---
  (1-b)  all clusters     = 0.8452   ->  b = 0.1548
  (1-b)  massive + local  = 0.8673   ->  b = 0.1327
  --- fix Y* only, jointly infer alpha,beta,(1-b) (beta free) ---
  alpha = 1.5699,  beta = 0.2962,  (1-b) = 0.8770  (b = 0.1230)
